# Higgs-portal homebrew shard validation

Read-only checks of the published homebrew hits, cells and particles: **100 events per pileup setting** by default. Pu0 is **blue**, pu200 **green**. The two settings use independent hard-scatter event selections; matching local IDs do not imply the same LLP event. The saved production report validates all 200 pu200 LLP decay endpoints.

Use **Restart Kernel and Run All** after changing paths. Existing saved outputs may describe the earlier one-event pilot until rerun. The full scan checks every event sequentially; only small count/coverage tables and flagged-hit rows are retained. No ACTS, Docker, PyTorch, ROOT inputs or further downloads are needed. These checks do not establish tracking efficiency or detector resolution.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "notebooks/validation/higgs_portal_validation.py").is_file())
sys.path.insert(0, str(REPO))
from notebooks.validation import higgs_portal_validation as validation
from notebooks.validation import paired_vs_published as paired

PAIRED_ROOT = Path("/mnt/data2/data/collider_ml/homebrew/higgs_portal_v2_cells")
# Alternatively: PAIRED_ROOT = REPO.parent / "exports/higgs-portal-pilot-run1-n1/paired"
EVENT = 0  # Detailed feature distributions, cells and galleries.
PREVIEW_EVENTS = (0, 24, 99)  # Lightweight spatial checks; no cell frames loaded.
RUN_FULL_SHARD_SCAN = True  # All 100 events, read sequentially. Can take several minutes.
PARTICLE_COHORT = "generator_charged_with_hits"
SCATTER_LIMIT = 15000
SEED = 7
GALLERY_SAMPLE = "pu200"
CLUSTER_TARGET_SIZES = (1, 2, 4, 8, 16, 32)
CHECK_SOURCE_HASHES = True

# Optional historical one-event numerical reference; unavailable for the new shard.
RUN_PERIGEE_REFERENCE = False
REFERENCE_PILOT_ROOT = REPO.parent / "exports/higgs-portal-pilot-run1-n1"
BASELINE_PAIRED_ROOT = REFERENCE_PILOT_ROOT / "paired.before-recovery-v2"
PERIGEE_VALIDATION_ROOT = REFERENCE_PILOT_ROOT / "validation/perigee_reference"

pd.set_option("display.max_columns", 30)
print("Parquet release being tested:", PAIRED_ROOT)
print("Selected local event:", EVENT, "in each independent sample")
print("Spatial preview events:", PREVIEW_EVENTS)
print("Full-shard validation enabled:", RUN_FULL_SHARD_SCAN)
before_hashes = validation.file_hashes(PAIRED_ROOT) if CHECK_SOURCE_HASHES else None

## Production evidence and event selection

Check completion markers and inspect the simulation gate and recovery provenance. The published pu0 ROOT input was already simulated; only pu200 was newly simulated in this pilot. LLP endpoint checks below concern that new pu200 simulation.

Particle `vx/vy/vz` are **production vertices**, not decay endpoints. An LLP's own production vertex should be near its collision vertex. Displaced daughter production vertices, perigee parameters, and the saved simulation endpoint check provide complementary evidence. `vertex_primary` is an event-local vertex identifier, not a boolean or guaranteed signal tag.

In [ ]:
pilot, complete, simulation_check = validation.production_evidence(PAIRED_ROOT)
display(pd.Series(pilot["physics"], name="Physics point").to_frame())
display(pd.Series({key: value for key, value in simulation_check.items()
                   if not isinstance(value, (list, dict))}, name="Simulation validation").to_frame())
display(pd.DataFrame(pilot["selected_signal_events"]).head(10))
if simulation_check.get("root_to_hepmc"):
    print("pu200 local-event to generator-event mapping (first ten entries):")
    display(pd.DataFrame(simulation_check["root_to_hepmc"]).rename(columns={"root_entry": "local_event"}).head(10))
print("Event pairing:", pilot["comparison_note"])
print("Release event count:", complete["events_per_sample"], "per setting; production run", pilot["run"])
print("Original source and validation checksums are in the provenance directory.")
boost_matches = pd.DataFrame(simulation_check.get("decay_boost_matches", []))
if not boost_matches.empty:
    display(boost_matches.head(20))

## Counts, schemas and associations

Loads only the selected event. Particle files are event-row/list-column tables, whereas hits and cells have one row per measurement or cell. Checks use the full measurement key:
`(campaign, dataset, version, run, local_event, measurement_id)`.

Every cell must have exactly one parent hit with the same geometry; every hit must have cells. Particle references and scalar labels are checked against this event's particles. Null scalar labels are permitted for multi-particle hits; contributor lists remain authoritative. Legacy converted-label disagreements and incomplete truth are reported separately, not hidden.

In [ ]:
samples = validation.load_samples(PAIRED_ROOT, EVENT)
analyses = {name: sample.analysis for name, sample in samples.items()}
checks = validation.integrity(samples)
display(checks)
display(validation.coverage(samples))
display(pd.DataFrame([
    dict(sample=name, measurements=sample.hits.num_rows, cells=sample.cells.num_rows,
         particles=len(sample.particles), particle_memory_MiB=sample.particles.memory_usage(deep=True).sum()/2**20,
         cell_analysis_memory_MiB=sample.analysis.cells.memory_usage(deep=True).sum()/2**20)
    for name, sample in samples.items()
]))
display(pd.concat([validation.truth_review(sample, limit=20) for sample in samples.values()], ignore_index=True))
print("Incomplete particle_ids are lower bounds; legacy converted labels are comparison diagnostics.")

## Missing and non-finite values

The table counts both nulls and NaN/infinity. Some perigee values are undefined, particularly for neutral particles or particles without a useful tracker trajectory. Null strip `pixel_code` values are intentional. These populations must not be silently filled or treated as pixel-encoding failures.

In [ ]:
nonfinite = validation.nonfinite_rates(samples)
display(nonfinite[nonfinite.nonfinite > 0])
rates = nonfinite.pivot(index=["table", "feature"], columns="sample", values="fraction").fillna(0)
rates = rates.loc[rates.max(axis=1).nlargest(16).index]
if rates.to_numpy().max() > 0:
    rates.plot.barh(figsize=(11, 6), color=[paired.PILEUP_COLORS[name] for name in rates.columns])
    plt.xlabel("Fraction non-finite / null")
    plt.title("Missing-value rates; strip pixel codes are intentionally null")
    plt.tight_layout()
    plt.show()
display(pd.DataFrame([
    dict(sample=name, cohort=cohort, particles=len(frame),
         finite_d0=int(np.isfinite(frame.perigee_d0).sum()),
         finite_z0=int(np.isfinite(frame.perigee_z0).sum()))
    for name, sample in samples.items()
    for cohort in ("all", "generator_charged_with_hits", "charged_with_hits", "displaced_charged_with_hits")
    for frame in [validation.particle_selection(sample.particles, cohort)]
]))

## Particle kinematics and displaced vertices

The default cohort is generator-origin charged particles (`primary == True`) with at least one **SimTrackerHit**. This excludes Geant4-created material secondaries, but is **not** a signal-only selection: pileup generator particles also qualify. The SimTrackerHit count need not equal the number of reconstructed measurements linked to a particle. Set `PARTICLE_COHORT` above to inspect all particles or the displaced subset (`production_r > 10 mm`), which also includes material secondaries.

Plots use full-event statistics, common bins and logarithmic density axes to expose tails. Categorical plots show the most common categories and retain full-event denominators. Momentum/energy are in GeV, positions/perigee in mm, time in ns.

In [ ]:
validation.plot_particles(samples, PARTICLE_COHORT)
llps = pd.concat([
    sample.particles.loc[sample.particles.pdg_id.abs() == 36,
        ["particle_id", "pdg_id", "mass", "pt", "px", "py", "pz", "vx", "vy", "vz", "production_r"]].assign(sample=name)
    for name, sample in samples.items()
], ignore_index=True)
display(llps)
display(pd.DataFrame([
    dict(sample=name, llps=int((sample.particles.pdg_id.abs() == 36).sum()),
         charged_with_hits=len(validation.particle_selection(sample.particles)),
         displaced_charged_with_hits=len(validation.particle_selection(sample.particles, "displaced_charged_with_hits")))
    for name, sample in samples.items()
]))

## Detector coverage and hit features

Production vertices and measurement positions use separate panels. Dense spatial views use deterministic sampling only; integrity counts and histograms use the complete selected events. Both pileup settings appear on the same axes. Hit `local0/local1` are signed sensor-local coordinates, not channel addresses.

No hit positions have been recomputed using quantized cell codes.

In [ ]:
validation.plot_spatial(samples, SCATTER_LIMIT, SEED, PARTICLE_COHORT)
validation.plot_hits(samples)

## Cells, cluster sizes and contributors

Activation remains the native ACTS path-length proxy in mm, after its configured noise/threshold/merging. A large activation does not by itself imply several particles. Particle multiplicity below belongs to the **parent cluster**, not to an individual cell. Incomplete truth is a separate category.

Cell and cluster denominators are reported separately. Full-range survival curves and largest-cell tables retain rare extremes.

In [ ]:
display(paired.pileup_event_summary(analyses))
display(paired.contributor_statistics(analyses))
display(paired.largest_cells(analyses, per_sample=10))
paired.plot_pileup_comparison(analyses)

## Frozen P99.9 pixel-code validation

The mapping was calibrated on ttbar pu200, not refitted to Higgs portal. Recompute every selected pixel code from the stored boundaries with upper-bin ties. Require positive codes 1-15 on pixels and null codes on strips.

Code 15 includes its nominal bin and true overflow. The table distinguishes all code-15 cells from values beyond the nominal code-15 bin end. This validates storage and frozen-map application, not electronics realism or a calibrated ATLAS response.

In [ ]:
display(validation.quantization(samples))
validation.plot_codes(samples)

## Example cluster images

Six different available cluster sizes per detector type, nearest to the configured targets, selected deterministically. Each gallery has a shared native-activation color scale. Images show actual channel addresses with one channel of padding; square drawn pixels do not imply square physical cells. There is no image-window cropping. Change `GALLERY_SAMPLE` to inspect the other sample.

In [ ]:
display(validation.cluster_gallery(samples[GALLERY_SAMPLE], CLUSTER_TARGET_SIZES))

## Several independent event displays

Inspect configured local events in both pileup settings. The second figure for each event shows measurements linked through `particle_ids` to displaced generator-origin charged particles with tracker hits. These are truth-linked measurements, **not reconstructed tracks** and not a signal-only selection; pileup generator particles can qualify. Unknown contributors are not invented.

Coordinates on particle plots are production vertices. Actual LLP decay endpoints are checked in the saved Geant4 validation report, not stored in the particle Parquet schema. Deterministic sampling affects spatial rendering only.

In [ ]:
available_events = set(range(complete["events_per_sample"]))
for preview_event in PREVIEW_EVENTS:
    if preview_event not in available_events:
        print("Skipping unavailable preview event", preview_event)
        continue
    print("Spatial preview: local event", preview_event, "in each independent sample")
    preview = validation.load_samples(PAIRED_ROOT, preview_event, include_cells=False)
    validation.plot_spatial(preview, SCATTER_LIMIT, SEED, PARTICLE_COHORT)
    validation.plot_linked_hits(preview, SCATTER_LIMIT, SEED)
    del preview

## All-event integrity, truth flags and quantization

This is a full scan, not a sample: each hits/cells shard is read once in increasing event order, alongside one nested particle row at a time. It checks full-key cell-to-hit joins, unique channel addresses, geometry, existing particle labels/contributors, event sets, frozen pixel encoding and null strip codes. Totals must reproduce `report.json`, including all unresolved links and legacy-label disagreements. Memory use is bounded by a single event plus input batches; source Parquets stay unchanged.

Truth limitations are distinct: an **incomplete list** is a lower bound because some SimHits could not be resolved uniquely; a **legacy-label disagreement** is a different result from production's coordinate matcher for a complete contributor list. It is not another unresolved SimHit. Use `particle_ids`, `particle_ids_complete` and `unresolved_simhit_ids` together. The table lists every flagged hit in the full scan, with lists retained. No flags are suppressed or automatically repaired.

Finite-perigee coverage is shown by cohort and event. A finite value is not an independent proof of numerical correctness; this does not replace the original numerical-reference study.

In [ ]:
campaign_scan = None
if RUN_FULL_SHARD_SCAN:
    campaign_scan = validation.scan_campaign(PAIRED_ROOT)
    event_counts = campaign_scan["counts"]
    display(event_counts.groupby("sample")[["measurements", "cells", "particles", "llps",
        "orphan_cells", "hits_without_cells", "missing_parent_ids", "unresolved_simhit_links",
        "incomplete_truth_measurements", "converted_label_disagreements"]].sum())
    display(event_counts)
    coverage_all = campaign_scan["perigees"].groupby(["sample", "cohort", "feature"])[
        ["particles", "finite", "nonfinite"]].sum()
    coverage_all["nonfinite_fraction"] = coverage_all.nonfinite / coverage_all.particles.replace(0, np.nan)
    display(coverage_all)
    display(campaign_scan["codes"])
    flagged_hits = campaign_scan["issues"]
    print("All flagged hits (complete scan):", len(flagged_hits))
    with pd.option_context("display.max_rows", 150, "display.max_colwidth", 80):
        display(flagged_hits)
    validation.plot_campaign(campaign_scan)
    print("PASS: all-event associations and quantization reproduce the production reports.")
else:
    print("Full-shard scan disabled: selected-event plots alone do not validate all 100 events.")

## Read-only confirmation and interpretation

The checks above establish file consistency, associations, feature availability, detector coverage and correct frozen encoding. They do **not** establish physical resolution, mass/lifetime reconstruction, downstream tracking efficiency, or event-matched pileup effects. Those require additional reconstruction and larger/matched samples.

Save this notebook after running it to retain figures and tables for review.

In [ ]:
if CHECK_SOURCE_HASHES:
    after_hashes = validation.file_hashes(PAIRED_ROOT)
    assert before_hashes == after_hashes, "Source Parquet contents changed during analysis"
    print("PASS: all six source Parquet SHA256 checksums are unchanged")
else:
    print("Source checksum verification was disabled")
print("Validation cells completed; review warnings, missingness and legacy-label disagreements above.")

## Current stored truth-perigee validation

This section tests the **current `PAIRED_ROOT` Parquet files** against the independent reference for the same event. The original reference is hash-checked against `BASELINE_PAIRED_ROOT`; all non-perigee particle features must match that baseline before its reference is reused. Current ROOT writer provenance and its input checksum are also checked.

The reference uses SciPy DOP853 Lorentz integration in the queried ODD field, independently of ACTS's stepper. The chosen branch is the local transverse-distance minimum nearest production in absolute path length, searching both directions within a finite bound. These are numerical truth-parameter checks, not detector resolution measurements.

Under the recovery-only policy, all original finite perigees must remain exactly unchanged. New recoveries must agree with the independent reference. Known legacy reference discrepancies are reported as `legacy_mismatch_preserved`; unresolved values remain NaN. Residuals below use actual values from the current Parquets, not the older diagnostic candidates. ROOT does not store the propagation path length, so no current path residual is claimed.


**Optional historical check:** disabled by default for the 100-event release. Its baseline ROOT files and independent references cover only the original one-event pilot. Enabling it for this new release does not supply a valid independent numerical reference. The full-shard section above checks finite d0/z0 coverage, not their numerical accuracy.

In [ ]:
if RUN_PERIGEE_REFERENCE:
    print("Current Parquet directory being tested:", PAIRED_ROOT)
    reference_summary, comparisons, uniform_checks, current_perigee_checks = validation.load_current_perigee_validation(
        samples, BASELINE_PAIRED_ROOT, PERIGEE_VALIDATION_ROOT)
    display(current_perigee_checks)
    display(pd.Series({key: reference_summary[key] for key in (
        "mode", "branch_convention", "reference", "input_precision", "tolerances_mm",
        "uniform_cases", "uniform_reference_pass", "source_files_unchanged")}).to_frame("Validation settings"))
    # These test the independent reference, not the old ACTS candidate outputs.
    display(uniform_checks[["case", "bz_T", "charge", "reference_pass",
                            "reference_delta_d0", "reference_delta_z0"]])
    display(validation.perigee_validation_statistics(comparisons))
    display(comparisons.loc[comparisons.status != "pass", [
        "sample", "particle_id", "pdg_id", "pt_GeV", "originally_finite",
        "method", "reference_status", "status", "candidate_d0", "candidate_z0",
        "delta_d0", "delta_z0"]])
else:
    print("Skipped: independent numerical perigee reference is specific to the original one-event pilot.")

In [ ]:
if RUN_PERIGEE_REFERENCE:
    validation.plot_perigee_validation(comparisons, reference_summary, value_label="Current stored perigee")
    display(comparisons.nlargest(10, "momentum_drift")[
        ["sample", "particle_id", "momentum_drift", "reference_curvature", "reference_status"]])
    checked = comparisons[comparisons.reference_converged.notna()]
    display(checked[["sample", "particle_id", "reference_converged", "refinement_levels",
                     "reference_delta_tighter_d0", "reference_delta_tighter_z0"]])
    assert reference_summary["source_files_unchanged"]
    if CHECK_SOURCE_HASHES:
        assert before_hashes == validation.file_hashes(PAIRED_ROOT)
    print("PASS: current Parquets tested; original finite perigees preserved; new recoveries validated; source files unchanged.")
else:
    print("Skipped: independent numerical perigee reference is specific to the original one-event pilot.")